In [48]:
import sys

sys.path.append('/home/mdafifal.mamun/research/S3M')

In [49]:
import json
import sys
from sparse_recommendation import RecommendationFileSparse


def convert_sparse_to_list(sparse_filepath):
    """
    Convert a .sparse file to python list.
    
    Args:
        sparse_filepath: Path to the input .sparse file
    """
    # Read the sparse file
    sparse_file = RecommendationFileSparse(None, sparse_filepath, -1, only_read=True)
    
    
    results = []
    for query_id, candidates, scores in sparse_file.read_file():
        # Convert to the format expected by RecommendationFileJson
        # Format: (report_id, candidates, scores)
        # Convert numpy types to Python native types for JSON serialization
        candidates_list = [int(c) for c in candidates]
        scores_list = [float(s) for s in scores]
        results.append((int(query_id), candidates_list, scores_list))
    
    return results

In [50]:
other_method_predictions = "/home/mdafifal.mamun/research/S3M/saved_predictions/tracesim_eclipse_results.sparse"
dedupt_predictions = "/home/mdafifal.mamun/research/S3M/saved_predictions/eclipse_dedupt_predictions_1771135140.5734932.json"


In [51]:
with open(dedupt_predictions, 'r') as f:
    dedupt_results = json.load(f)

In [52]:
dedupt_processed_results = []

for _, result in dedupt_results.items():
    bug_id = result["bug_id"]
    actual_duplicate_id = result["actual_duplicate_id"]
    preds = [int(pred) for pred in result["predictions"].keys()]

    dedupt_processed_results.append((bug_id, actual_duplicate_id, preds))
    

In [53]:
other_method_results = convert_sparse_to_list(other_method_predictions)

In [54]:
other_method_processed_results = []

for bug_id, candidates, scores in other_method_results:
    other_method_processed_results.append((bug_id, candidates[:25]))

In [55]:
# Find common bug IDs between the two methods
common_bug_ids = set([result[0] for result in dedupt_processed_results]) & set([result[0] for result in other_method_processed_results])

# Keep only the results for the common bug IDs
dedupt_common_results = [result for result in dedupt_processed_results if result[0] in common_bug_ids]
other_method_common_results = [result for result in other_method_processed_results if result[0] in common_bug_ids]

In [56]:
len(dedupt_common_results)

2146

In [57]:
# Create a combined dataframe from the common results
import pandas as pd

combined_results = []

for dedupt_result in dedupt_common_results:
    bug_id, actual_duplicate_id, dedupt_preds = dedupt_result
    
    # Find the corresponding other method result
    other_method_result = next((result for result in other_method_common_results if result[0] == bug_id), None)
    
    if other_method_result is not None:
        _, other_method_preds = other_method_result
        combined_results.append({
            "bug_id": bug_id,
            "actual_duplicate_id": actual_duplicate_id,
            "dedupt_preds": dedupt_preds,
            "other_method_preds": other_method_preds
        })
combined_df = pd.DataFrame(combined_results)

In [58]:
import pandas as pd

def reciprocal_rank(preds, actual):
    try:
        rank = preds.index(actual) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0


combined_df["rr_dedupt"] = combined_df.apply(
    lambda x: reciprocal_rank(x["dedupt_preds"], x["actual_duplicate_id"]),
    axis=1
)

combined_df["rr_other"] = combined_df.apply(
    lambda x: reciprocal_rank(x["other_method_preds"], x["actual_duplicate_id"]),
    axis=1
)


In [59]:
combined_df

,bug_id,actual_duplicate_id,dedupt_preds,other_method_preds,rr_dedupt,rr_other
0,416153,407502,"[407502, 395935, 402412, 384263, 415789, 40991...","[415946, 407502, 409876, 411851, 400538, 36763...",1.0,0.500000
1,416155,407502,"[384263, 407502, 405921, 395935, 389685, 38953...","[411851, 415946, 409876, 407502, 416153, 38426...",0.5,0.250000
2,416273,410011,"[410011, 399690, 404388, 393054, 410509, 41277...","[410011, 412518, 398324, 392561, 377593, 40270...",1.0,1.000000
3,416382,390409,"[390409, 411208, 390072, 405821, 394344, 39420...","[390409, 369663, 373897, 358774, 355722, 35571...",1.0,1.000000
4,416383,390409,"[390409, 411208, 390072, 405821, 404162, 39420...","[369245, 415925, 373298, 415512, 366785, 37803...",1.0,0.000000
...,...,...,...,...,...,...
2141,473716,428815,"[428815, 469068, 468909, 462122, 464563, 46781...","[428815, 471869, 473059, 464563, 469076, 46890...",1.0,1.000000
2142,473725,473289,"[473289, 463057, 462819, 462826, 462839, 46294...","[473289, 462819, 463057, 439811, 419294, 47119...",1.0,1.000000
2143,473753,465898,"[465898, 452583, 466908, 467453, 444919, 44494...","[465898, 467169, 466681, 459649, 467453, 46211...",1.0,1.000000
2144,473775,458689,"[461264, 443861, 443873, 444193, 452567, 46426...","[458689, 467954, 467378, 461472, 446357, 45188...",0.0,1.000000


In [60]:
from scipy.stats import wilcoxon

stat, p_value = wilcoxon(
    combined_df["rr_dedupt"],
    combined_df["rr_other"],
    alternative="two-sided"
)

print(f"Wlicoxon stat: {stat}, P-value: {p_value}")
if p_value < 0.001:
    print(f"    Result: *** Highly significant (p < 0.001)")
elif p_value < 0.01:
    print(f"    Result: ** Very significant (p < 0.01)")
elif p_value < 0.05:
    print(f"    Result: * Significant (p < 0.05)")
else:
    print(f"    Result: Not significant (p >= 0.05)")


Wlicoxon stat: 145951.0, P-value: 2.1188122857259064e-158
    Result: *** Highly significant (p < 0.001)
